# ENGG2112 — Fall Detection: Image-Level Model Pipeline
Baseline CNN · MobileNetV2 · EfficientNetB0 · Comparison · Plot Saving

## 2. Mount Google Drive and Set Plot Save Directory

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

SAVE_DIR = '/content/drive/MyDrive/report_engg2112_plots'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f'Plots will be saved to: {SAVE_DIR}')

## 1. Install and Import Libraries

In [ ]:
!pip install -q opendatasets pillow

import os, json, random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2, EfficientNetB0
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix, classification_report,
    roc_curve, precision_recall_curve, auc,
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
)

tf.random.set_seed(42)
np.random.seed(42)
print('Libraries loaded.')

## 3. Download Datasets

In [ ]:
import opendatasets as od

od.download('https://www.kaggle.com/datasets/jonathannield/cctv-action-recognition-dataset')
od.download('https://www.kaggle.com/datasets/uttejkumarkandagatla/fall-detection-dataset')

DATASET_1 = Path('/content/fall-detection-dataset/fall_dataset')
print('Dataset 1 exists:', DATASET_1.exists())

## 4. Data Pre-Processing and Cleaning
Verify YOLO label integrity and build a clean binary manifest from Dataset 1.

In [ ]:
def filename_group(stem):
    s = stem.lower()
    if s.startswith('fall'): return 'fall_file'
    elif s.startswith('not fallen'): return 'not_fallen_file'
    return 'unknown'

def image_label_purity(label_dir):
    rows = []
    for lf in sorted(label_dir.glob('*.txt')):
        with open(lf, 'r') as f:
            lines = [l.strip() for l in f if l.strip()]
        class_ids = sorted(set(int(float(l.split()[0])) for l in lines))
        stem = lf.stem.lower()
        expected_group = filename_group(lf.stem)
        if expected_group == 'fall_file':
            is_pure = set(class_ids).issubset({0})
        elif expected_group == 'not_fallen_file':
            is_pure = set(class_ids).issubset({1, 2})
        else:
            is_pure = False
        rows.append({'file': lf.name, 'stem': lf.stem,
                     'filename_group': expected_group,
                     'class_ids_present': class_ids,
                     'n_unique_classes': len(class_ids),
                     'is_pure_for_filename_group': is_pure})
    return pd.DataFrame(rows)

purity_train = image_label_purity(DATASET_1 / 'labels' / 'train')
purity_val   = image_label_purity(DATASET_1 / 'labels' / 'val')

print('TRAIN purity summary:')
display(purity_train.groupby(['filename_group','is_pure_for_filename_group']).size().reset_index(name='count'))
print('\nVAL purity summary:')
display(purity_val.groupby(['filename_group','is_pure_for_filename_group']).size().reset_index(name='count'))

In [ ]:
def build_clean_manifest(image_root, purity_df):
    rows = []
    for _, row in purity_df.iterrows():
        if not row['is_pure_for_filename_group']: continue
        stem = Path(row['file']).stem
        group = row['filename_group']
        binary_label = 'fall' if group == 'fall_file' else 'not_fallen'
        for ext in ['.jpg', '.jpeg', '.png']:
            img_path = image_root / (stem + ext)
            if img_path.exists():
                rows.append({'image_path': str(img_path), 'image_name': stem+ext,
                             'binary_label': binary_label})
                break
    return pd.DataFrame(rows)

train_manifest = build_clean_manifest(DATASET_1/'images'/'train', purity_train)
val_manifest   = build_clean_manifest(DATASET_1/'images'/'val',   purity_val)

train_manifest['split'] = 'train'
val_manifest['split']   = 'val'
clean_manifest = pd.concat([train_manifest, val_manifest], ignore_index=True)

print('Clean binary label counts:')
display(clean_manifest.groupby(['split','binary_label']).size().reset_index(name='count'))

## 5. Load Dataset Arrays

In [ ]:
TARGET_SIZE = (224, 224)
label_map = {'fall': 1, 'not_fallen': 0}

def load_dataset_arrays(df, split, target_size=TARGET_SIZE):
    subset = df[df['split'] == split].copy()
    X, y = [], []
    for _, row in subset.iterrows():
        img = Image.open(row['image_path']).convert('RGB')
        img = img.resize(target_size)
        X.append(np.array(img).astype('float32') / 255.0)
        y.append(label_map[row['binary_label']])
    return np.array(X), np.array(y)

X_train, y_train = load_dataset_arrays(clean_manifest, 'train')
X_val,   y_val   = load_dataset_arrays(clean_manifest, 'val')

print('X_train shape:', X_train.shape)
print('X_val shape:  ', X_val.shape)
print('Train label counts:', np.bincount(y_train))
print('Val label counts:  ', np.bincount(y_val))

classes = np.unique(y_train)
weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train)
class_weight = {int(c): float(w) for c, w in zip(classes, weights)}
print('Class weights:', class_weight)

## 6. Baseline CNN

In [ ]:
model_base = keras.Sequential([
    layers.Input(shape=(224, 224, 3)),
    layers.Conv2D(16, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),
    layers.Conv2D(32, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),
    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(1, activation='sigmoid')
])

model_base.compile(
    optimizer='adam', loss='binary_crossentropy',
    metrics=['accuracy',
             keras.metrics.Precision(name='precision'),
             keras.metrics.Recall(name='recall')]
)

early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

history_base = model_base.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=15, batch_size=16,
    class_weight=class_weight,
    callbacks=[early_stop], verbose=1
)

y_prob_base = model_base.predict(X_val).ravel()
print('Baseline CNN training complete.')

## 7. MobileNetV2 Transfer Learning

In [ ]:
base_model_v2 = MobileNetV2(input_shape=(224,224,3), include_top=False, weights='imagenet')
base_model_v2.trainable = False

x = base_model_v2.output
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(64, activation='relu')(x)
x = layers.Dropout(0.3)(x)
output = layers.Dense(1, activation='sigmoid')(x)

model_v2 = models.Model(inputs=base_model_v2.input, outputs=output)
model_v2.compile(optimizer='adam', loss='binary_crossentropy',
                 metrics=['accuracy', 'precision', 'recall'])

early_stop_v2 = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

history_v2 = model_v2.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=10, batch_size=16,
    class_weight=class_weight,
    callbacks=[early_stop_v2], verbose=1
)

y_prob_v2 = model_v2.predict(X_val).ravel()
print('MobileNetV2 training complete.')

## 8. EfficientNetB0 Transfer Learning

In [ ]:
from tensorflow.keras.applications.efficientnet import preprocess_input as eff_preprocess

X_train_eff = eff_preprocess(X_train.copy() * 255.0)
X_val_eff   = eff_preprocess(X_val.copy()   * 255.0)

base_model_eff = EfficientNetB0(input_shape=(224,224,3), include_top=False, weights='imagenet')
base_model_eff.trainable = True
for layer in base_model_eff.layers[:-20]:
    layer.trainable = False

x = base_model_eff.output
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.5)(x)
output = layers.Dense(1, activation='sigmoid')(x)

model_eff = models.Model(inputs=base_model_eff.input, outputs=output)
model_eff.compile(
    optimizer=keras.optimizers.Adam(1e-5),
    loss='binary_crossentropy',
    metrics=['accuracy', 'precision', 'recall']
)

early_stop_eff = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

history_eff = model_eff.fit(
    X_train_eff, y_train,
    validation_data=(X_val_eff, y_val),
    epochs=15, batch_size=16,
    class_weight=class_weight,
    callbacks=[early_stop_eff], verbose=1
)

y_prob_eff = model_eff.predict(X_val_eff).ravel()
print('EfficientNetB0 training complete.')

## 9. Helper: Save Plot to Drive

In [ ]:
def save_plot(filename):
    path = os.path.join(SAVE_DIR, filename)
    plt.savefig(path, dpi=150, bbox_inches='tight')
    print(f'Saved: {path}')

## 10. Confusion Matrices — All Three Models
Saved as `all_confusion_matrices.png` for the report.

In [ ]:
THRESHOLD_BASE = 0.24
THRESHOLD_V2   = 0.50
THRESHOLD_EFF  = 0.50

y_pred_base = (y_prob_base >= THRESHOLD_BASE).astype(int)
y_pred_v2   = (y_prob_v2   >= THRESHOLD_V2).astype(int)
y_pred_eff  = (y_prob_eff  >= THRESHOLD_EFF).astype(int)

cms = [
    (confusion_matrix(y_val, y_pred_base), f'Baseline CNN\n(threshold = {THRESHOLD_BASE})'),
    (confusion_matrix(y_val, y_pred_v2),   f'MobileNetV2\n(threshold = {THRESHOLD_V2})'),
    (confusion_matrix(y_val, y_pred_eff),  f'EfficientNetB0\n(threshold = {THRESHOLD_EFF})'),
]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
labels = ['not_fallen', 'fall']

for ax, (cm, title) in zip(axes, cms):
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=labels, yticklabels=labels, cbar=False)
    ax.set_title(title, fontsize=11)
    ax.set_xlabel('Predicted Label')
    ax.set_ylabel('True Label')

plt.suptitle('Confusion Matrices — Image-Level Models', fontsize=13, y=1.02)
plt.tight_layout()
save_plot('all_confusion_matrices.png')
plt.show()

## 11. ROC and Precision-Recall Curves — Baseline CNN
Saved as `roc_pr_curves.png` for the report.

In [ ]:
fpr_b, tpr_b, thresh_b = roc_curve(y_val, y_prob_base)
roc_auc_b = auc(fpr_b, tpr_b)
prec_b, rec_b, _ = precision_recall_curve(y_val, y_prob_base)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# ROC
axes[0].plot(fpr_b, tpr_b, linewidth=2, label=f'AUC = {roc_auc_b:.4f}')
axes[0].plot([0,1],[0,1], linestyle='--', color='orange', label='Random baseline')
thresh_idx = np.argmin(np.abs(thresh_b - THRESHOLD_BASE))
axes[0].axvline(x=fpr_b[thresh_idx], color='r', linestyle='--',
                label=f'Threshold = {THRESHOLD_BASE}')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate (Recall)')
axes[0].set_title('ROC Curve — Baseline CNN')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# PR
axes[1].plot(rec_b, prec_b, linewidth=2)
axes[1].axvline(x=0.956, color='r', linestyle='--', label='Operating point (recall=0.956)')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve — Baseline CNN')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
save_plot('roc_pr_curves.png')
plt.show()
print(f'Baseline CNN AUC-ROC: {roc_auc_b:.4f}')

## 12. ROC Curves — All Three Models Compared
Saved as `roc_all_models.png` for the report.

In [ ]:
fpr_v2, tpr_v2, _ = roc_curve(y_val, y_prob_v2)
fpr_ef, tpr_ef, _ = roc_curve(y_val, y_prob_eff)
roc_auc_v2 = auc(fpr_v2, tpr_v2)
roc_auc_ef = auc(fpr_ef, tpr_ef)

plt.figure(figsize=(7, 5))
plt.plot(fpr_b,  tpr_b,  linewidth=2, label=f'Baseline CNN     AUC = {roc_auc_b:.4f}')
plt.plot(fpr_v2, tpr_v2, linewidth=2, label=f'MobileNetV2      AUC = {roc_auc_v2:.4f}')
plt.plot(fpr_ef, tpr_ef, linewidth=2, label=f'EfficientNetB0   AUC = {roc_auc_ef:.4f}')
plt.plot([0,1],[0,1], linestyle='--', color='gray', alpha=0.5, label='Random baseline')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate (Recall)')
plt.title('ROC Curves — All Models')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
save_plot('roc_all_models.png')
plt.show()

## 13. Model Comparison Table
Displays and saves the full metrics comparison used in the report.

In [ ]:
comparison = pd.DataFrame([
    {
        'Model': 'Baseline CNN', 'Threshold': THRESHOLD_BASE,
        'Accuracy':        round(accuracy_score(y_val, y_pred_base), 4),
        'Fall Precision':  round(precision_score(y_val, y_pred_base), 4),
        'Fall Recall':     round(recall_score(y_val, y_pred_base), 4),
        'Fall F1':         round(f1_score(y_val, y_pred_base), 4),
        'AUC-ROC':         round(roc_auc_score(y_val, y_prob_base), 4),
    },
    {
        'Model': 'MobileNetV2', 'Threshold': THRESHOLD_V2,
        'Accuracy':        round(accuracy_score(y_val, y_pred_v2), 4),
        'Fall Precision':  round(precision_score(y_val, y_pred_v2), 4),
        'Fall Recall':     round(recall_score(y_val, y_pred_v2), 4),
        'Fall F1':         round(f1_score(y_val, y_pred_v2), 4),
        'AUC-ROC':         round(roc_auc_score(y_val, y_prob_v2), 4),
    },
    {
        'Model': 'EfficientNetB0', 'Threshold': THRESHOLD_EFF,
        'Accuracy':        round(accuracy_score(y_val, y_pred_eff), 4),
        'Fall Precision':  round(precision_score(y_val, y_pred_eff), 4),
        'Fall Recall':     round(recall_score(y_val, y_pred_eff), 4),
        'Fall F1':         round(f1_score(y_val, y_pred_eff), 4),
        'AUC-ROC':         round(roc_auc_score(y_val, y_prob_eff), 4),
    },
])

display(comparison)

# Save as CSV to Drive
csv_path = os.path.join(SAVE_DIR, 'model_comparison.csv')
comparison.to_csv(csv_path, index=False)
print(f'Saved: {csv_path}')

## 14. Full Classification Reports

In [ ]:
for name, y_pred, y_prob in [
    ('Baseline CNN',   y_pred_base, y_prob_base),
    ('MobileNetV2',    y_pred_v2,   y_prob_v2),
    ('EfficientNetB0', y_pred_eff,  y_prob_eff),
]:
    print(f'=== {name} ===')
    print(classification_report(y_val, y_pred,
                                target_names=['not_fallen','fall'], digits=4))
    print(f'AUC-ROC: {roc_auc_score(y_val, y_prob):.4f}\n')

## 15. Confirm All Files Saved to Drive

In [ ]:
print(f'Files in {SAVE_DIR}:')
for f in sorted(os.listdir(SAVE_DIR)):
    size = os.path.getsize(os.path.join(SAVE_DIR, f)) / 1024
    print(f'  {f:40s} {size:.1f} KB')